## Scenario: Course Inbox Automation for a University Class

You’re a **course team member / TA / quality coordinator** handling student emails such as:
- Extension requests  
- Late submissions  
- Plagiarism concerns  
- Recheck complaints  

Students often send follow-up emails like:

> *“Please extend again, I still need more time.”*

---

## The Problem

A **stateless agent** (no memory) cannot determine whether a student has already received an extension earlier.

A **stateful agent** (with memory) can enforce policies such as:

> **“Only one extension per assignment.”**

---

## The Solution (Demo Overview)

This demo builds a small **ReAct-style agent** that:

1. **Looks up policy** from a tiny knowledge base  
2. **Drafts a professional response email**  
3. **Makes a final decision**  
4. **Demonstrates how memory changes the outcome**  

---

## Key Learning Point

Memory transforms an agent from:
-  Repeating decisions blindly  
-  Enforcing rules consistently and fairly  


## Prerequisites

Make sure Ollama is running locally:
```bash
ollama pull qwen2.5:1.5b
ollama serve
```

Then run the cells below.

In [10]:
import requests
import json
import re
from typing import Optional, List, Dict, Callable
from dataclasses import dataclass, field

### LLM (Ollama) configuration + call function

In [11]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen2.5:1.5b"

def call_llm(prompt: str) -> str:
    """Call Ollama and get response."""
    r = requests.post(
        OLLAMA_URL,
        json={"model": MODEL, "prompt": prompt, "stream": False},
        timeout=120
    )
    r.raise_for_status()
    return r.json()["response"]

print("LLM Ready:", MODEL)

LLM Ready: qwen2.5:1.5b


In [12]:
# !pip install groq


In [13]:
# from groq import Groq
# from getpass import getpass

# # Secure runtime input (not vQisible on screen)
# api_key = getpass("Enter your Groq API key: ")

# MODEL = "llama-3.1-8b-instant"
# client = Groq(api_key=api_key)

# def call_llm(prompt: str) -> str:
#     completion = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "user", "content": prompt}
#         ],
#         temperature=0,     # keep deterministic
#         max_tokens=800
#     )
#     return completion.choices[0].message.content


### Tools (mini knowledge base + email generator)

In [20]:
KB = {
    "extension_policy": "Extensions are granted once per assignment for valid reasons. Repeated extension requests require instructor approval.",
    "late_penalty": "Late penalty is 5 marks per day (max 20). After 3 late days, escalate.",
    "plagiarism_policy": "If plagiarism > 30%, request resubmission. If repeated, escalate.",
    "recheck_policy": "Grade recheck requires specific questions. Repeated complaints escalate."
}


In [21]:
def kb_lookup(key: str) -> str:
    return KB.get(key.strip(), "Policy not found.")

In [22]:
def draft_email(args_json: str) -> str:
    """
    Tool expects JSON string:
    {
      "student_id": "...",
      "subject": "...",
      "decision": "...",
      "rationale": "...",
      "next_steps": "..."
    }
    """
    try:
        args = json.loads(args_json)
    except Exception:
        return "error: invalid JSON"

    student = args.get("student_id", "(unknown)")
    subject = args.get("subject", "Course Request")
    decision = args.get("decision", "(pending)")
    rationale = args.get("rationale", "")
    next_steps = args.get("next_steps", "")

    return (
        f"Subject: {subject}\n\n"
        f"Hi {student},\n\n"
        f"{decision}\n\n"
        f"Reason: {rationale}\n\n"
        f"Next steps: {next_steps}\n\n"
        f"Regards,\nCourse Team"
    )

TOOLS: Dict[str, Callable] = {
    "kb_lookup": kb_lookup,
    "draft_email": draft_email
}

print("Tools ready:", ", ".join(TOOLS.keys()))

Tools ready: kb_lookup, draft_email


### Memory Store (state)

This class stores past actions for each student so the agent can remember what happened before.
It lets the agent add new events and retrieve history whenever needed.

This dictionary keeps a timeline of events for each student:

```python
memory = {
    "22k-1234": [
        {"intent": "extension", "assignment": "A2", "action": "GRANT_EXTENSION"},
        {"intent": "extension", "assignment": "A2", "action": "ASK_CLARIFICATION"},
        {"intent": "resubmission", "assignment": "A2", "action": "APPROVED"}
    ],
    "22k-5678": [
        {"intent": "extension", "assignment": "A1", "action": "GRANT_EXTENSION"},
        {"intent": "late_submission", "assignment": "A1", "action": "PENALTY_APPLIED"}
    ]
}


In [23]:
@dataclass
class MemoryStore:
    """Store student event history."""
    student_events: Dict[str, List[dict]] = field(default_factory=dict)

    def add(self, student_id: str, event: dict):
#         create-if-missing + store event.
        self.student_events.setdefault(student_id, []).append(event)

    def history(self, student_id: str) -> List[dict]:
        return self.student_events.get(student_id, [])

mem = MemoryStore()

#### Preload memory: student already got an extension

In [24]:
mem.add("22k-1234", {
    "intent": "extension",
    "assignment": "A2",
    "message_id": "m1",
    "action": "GRANT_EXTENSION"
})

#### Incoming email test case

In [25]:
email_followup = {
    "student_id": "22k-1234",
    "message_id": "m2",
    "intent": "extension",
    "assignment": "A2",
    "late_days": 2,
    "text": "Please extend again, I still need more time."
#         "text": "my plagiarism is 30 I need a chance."
    
}


### System instruction (ReAct protocol)

In [26]:
SYSTEM_INSTRUCTION = """You are a ReAct agent for course-inbox automation.

STRICT FORMAT — Exactly 3 turns, and EXACTLY ONE ACTION PER TURN.

Turn 1: Look up policy
Thought: <one sentence>
Action: kb_lookup[extension_policy]  
(or late_penalty, or plagiarism_policy, or recheck_policy)

Turn 2: Draft email
Thought: <one sentence>
Action: draft_email[<valid JSON string>]

Turn 3: Final decision
Thought: <one sentence>
Final Answer:
{
  "action": "...",
  "rationale": "...",
  "next_steps": "..."
}

VALID ACTIONS:
GRANT_EXTENSION, ESCALATE, ASK_CLARIFICATION, ADVISE_REWRITE

--------------------------------------------------
MEMORY RULE
--------------------------------------------------
If MEMORY is provided and shows action = "GRANT_EXTENSION" for the SAME assignment,
then the student has already used their extension.
You MUST NOT grant another extension.
Choose ESCALATE or ADVISE_REWRITE.

--------------------------------------------------
ANTI-HALLUCINATION RULE (CRITICAL)
--------------------------------------------------
1. If MEMORY is NOT provided:
   - You MUST NOT claim that the student previously received an extension.
   - You MUST NOT assert confirmed history of prior approvals or denials.

2. You MAY infer that this is a repeat request ONLY if the EMAIL TEXT itself
   explicitly indicates it (e.g., words like "again", "as requested earlier").

3. In stateless mode:
   - If the email suggests a repeat request, choose ESCALATE or ASK_CLARIFICATION.
   - Your rationale MUST explain uncertainty (lack of verified history),
     not assert facts about past actions.

4. NEVER fabricate student history.
   - Do not use phrases such as:
     "already received", "previously granted", "second extension"
     unless MEMORY explicitly confirms it.

--------------------------------------------------
REASONING CONSTRAINT
--------------------------------------------------
All reasoning and decisions must be grounded ONLY in:
- The current email text
- The retrieved policy
- The provided MEMORY block (if present)

If information is missing, express uncertainty explicitly.
Do NOT invent facts to complete the reasoning.


"""

### Helper functions (parsing + safety)

This function scans the agent’s reasoning text and extracts the most recent tool call (which tool was used and with what input).

It lets our ReAct agent know what action it decided last, so the system can execute that tool next.

In [27]:
def _last_action(text: str):
    """
    Finds the last occurrence of:
    Action: tool_name[...]
    Returns (tool_name, argument_text_inside_brackets)
    """
    matches = list(re.finditer(r"Action:\s*([a-zA-Z_]\w*)\[(.*?)\]", text, re.DOTALL))
    if not matches:
        return None, None
    m = matches[-1]
    return m.group(1).strip(), m.group(2).strip()

* Thought...
* Action: kb_lookup[extension_policy]
* Thought...
* Action: draft_email[...]

This function grabs:

* draft_email
* its input


##### This function pulls out the first complete JSON response from the agent’s text, even if there’s extra reasoning around it.

In [28]:
def extract_first_json(text: str):
    """Extract the first {...} JSON object from text (balanced braces)."""
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

### Typical LLM Response (Unstructured)

LLMs often reply like this:

```text
Thought: ...
Here is my decision:
{ "action": "GRANT_EXTENSION", "rationale": "..." }
More text...

using this function we get this

{
  "action": "...",
  "rationale": "..."
}


#### This function tries to convert the agent’s JSON text into a Python dictionary safely

In [29]:
def _parse_json_obj(jtxt: Optional[str]) -> dict:
    if not jtxt:
        return {}
    try:
        obj = json.loads(jtxt)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}

#### This function cleans the tool input and makes sure the agent only queries valid knowledge-base keys. (tool sanitization)

In [30]:
def _safe_kb_key(arg: str) -> str:
    arg = (arg or "").strip()
    return arg if arg in KB else "extension_policy"

#### This function translates the agent’s internal decision into a human-friendly email subject and message.

In [31]:
def _decision_to_email_fields(action: str, assignment: str) -> Dict[str, str]:
    """Convert action to a human email 'decision' line + subject."""
    subject = f"{assignment} — Extension Request Update"
    if action == "GRANT_EXTENSION":
        decision = "Your extension request has been approved."
    elif action == "ASK_CLARIFICATION":
        decision = "I need one clarification before I can process your request."
    elif action == "ADVISE_REWRITE":
        decision = "I can’t approve another extension for the same assignment."
    elif action == "ESCALATE":
        decision = "I’m escalating this request to the instructor for review."
    else:
        decision = "I’m unable to process this request at this time."
    return {"subject": subject, "decision": decision}



### Deterministic memory check

#### This function gives the agent long-term consistency by checking past decisions before acting.

In [32]:
def has_prior_extension(memory_context: Optional[str], assignment: str) -> bool:
    """Deterministically check memory for a prior GRANT_EXTENSION on the same assignment."""
    if not memory_context or not str(memory_context).strip():
        return False
    try:
        hist = json.loads(memory_context)
        if not isinstance(hist, list):
            return False
        for ev in hist:
            if ev.get("assignment") == assignment and ev.get("action") == "GRANT_EXTENSION":
                return True
    except Exception:
        return False
    return False

### ReAct Agent Loop

In [33]:
def react_agent(email: dict, memory_context: Optional[str] = None, verbose: bool = True):
    # ---------------------------
    # Stateless rationale sanitizer
    # ---------------------------
    
#     Stateless rationale sanitizer (prevent lying when no memory)
    def sanitize_stateless_rationale(rationale: str) -> str:
        r = (rationale or "").strip()

        forbidden = [
            r"\balready received\b",
            r"\bpreviously received\b",
            r"\bwas granted\b",
            r"\bhas been granted\b",
            r"\bgranted (an|a) extension\b",
            r"\balready granted\b",
            r"\bsecond extension\b",
            r"\balready used\b",
            r"\bused their extension\b",
            r"\bextension already used\b",
            r"\bonce already\b",
        ]
        for pat in forbidden:
            r = re.sub(pat, "", r, flags=re.IGNORECASE)

        r = re.sub(r"\s+", " ", r).strip()
        return r

    # ---------------------------
    # Context preparation
    # ---------------------------
#     memory must exist AND must not be empty after trimming
    has_memory = bool(memory_context and str(memory_context).strip())
    
#     Adds memory into prompts only if it exists.
    memory_block = f"\n=== MEMORY (use to decide) ===\n{memory_context}\n" if has_memory else ""
    
#     Formats the email nicely for the LLM and creates a trace log.
    email_block = "Current email:\n" + json.dumps(email, indent=2)
    trace: List[str] = []

    assignment = email.get("assignment", "") or ""
    # (email_text is not strictly needed here, but kept for clarity)
    email_text = email.get("text", "") or ""

    # ---------------------------
    # TURN 1: Policy Lookup (ONE tool)
    # ---------------------------
    p1 = f"""{SYSTEM_INSTRUCTION}

TURN 1/3 - POLICY LOOKUP
{memory_block}
{email_block}

You MUST output:
Thought: <one sentence>
Action: kb_lookup[extension_policy] OR kb_lookup[late_penalty] OR kb_lookup[plagiarism_policy] OR kb_lookup[recheck_policy]
(End your message with exactly ONE Action line.)
"""
    out1 = call_llm(p1).strip()
    trace.append(out1)
    if verbose:
        print("\n--- Step 1 Policy LookUp ---\n", out1[:300])
        
#     Extracts the tool name + key from the LLM’s last Action: ...[...].

    tool1, arg1 = _last_action(out1)

    # Enforce kb_lookup + valid key
    if tool1 != "kb_lookup":
        arg1 = "extension_policy"
    arg1 = (arg1 or "").strip()
    if arg1 not in KB:
        arg1 = "extension_policy"

    policy = TOOLS["kb_lookup"](arg1)

    # ---------------------------
    # TURN 3: Final Decision (STRICT JSON) — LLM decides (NO forced action)
    # ---------------------------
    stateless_line = ""
    if not has_memory:
        stateless_line = (
            'Stateless constraint: Your rationale MUST include this exact sentence: '
            '"I do not have access to prior history in this run."\n'
        )

    p3 = f"""{SYSTEM_INSTRUCTION}

TURN 3/3 - FINAL DECISION (STRICT OUTPUT)

{stateless_line}
Return ONLY valid JSON (no extra text, no markdown).
Required schema:
{{
  "action": "GRANT_EXTENSION" | "ASK_CLARIFICATION" | "ESCALATE" | "ADVISE_REWRITE",
  "rationale": "1-3 sentences grounded ONLY in policy + email + memory (if provided).",
  "next_steps": "1-3 sentences describing what staff should do next."
}}

Policy (from tools): {policy}

{memory_block}
{email_block}
"""
    out3 = call_llm(p3).strip()
    trace.append(out3)
    if verbose:
        print("\n--- Step 2 Decision ---\n", out3[:300])

    decision_obj = _parse_json_obj(extract_first_json(out3))

    # Repair if missing required fields
    required = {"action", "rationale", "next_steps"}
    if not required.issubset(decision_obj.keys()):
        repair_prompt = f"""{SYSTEM_INSTRUCTION}

You returned JSON missing required keys.
Return ONLY corrected JSON with keys: action, rationale, next_steps.

Previous output:
{out3}
"""
        out3b = call_llm(repair_prompt).strip()
        trace.append(out3b)
        if verbose:
            print("\n--- Step 2b (repair) ---\n", out3b[:300])
        decision_obj = _parse_json_obj(extract_first_json(out3b))

    # Fallbacks
    action = decision_obj.get("action") or "ASK_CLARIFICATION"
    rationale = decision_obj.get("rationale") or "Policy-based decision."
    next_steps = decision_obj.get("next_steps") or "Proceed per policy."

    # ---------------------------
    # Truthfulness enforcement in STATELESS mode
    # ---------------------------
    if not has_memory:
        disclaimer = "I do not have access to prior history in this run."
        if disclaimer not in rationale:
            rationale = f"{disclaimer} {rationale}".strip()
        rationale = sanitize_stateless_rationale(rationale)

    # ---------------------------
    # Memory audit (NO enforcement) — warn if LLM conflicts with confirmed history
    # ---------------------------
    # If memory shows prior GRANT_EXTENSION for same assignment AND LLM chose GRANT_EXTENSION,
    # we do NOT override, but we annotate rationale so staff sees the conflict.
    prior_ext = has_prior_extension(memory_context, assignment)
    if prior_ext and action == "GRANT_EXTENSION":
        rationale = (
            "Memory indicates a prior extension was granted for this assignment; "
            "this selected action may conflict with the one-extension policy. "
            + rationale
        ).strip()

    # ---------------------------
    # TURN 2: Draft Email (deterministic mapping + tool execution)
    # ---------------------------
    subject = f"{assignment} — Extension Request Update" if assignment else "Extension Request Update"

    if action == "GRANT_EXTENSION":
        decision_line = "Your extension request has been approved."
    elif action == "ASK_CLARIFICATION":
        decision_line = "I need one clarification before I can process your request."
    elif action == "ADVISE_REWRITE":
        decision_line = "I can’t approve another extension for the same assignment; here are your options."
    elif action == "ESCALATE":
        decision_line = "I’m escalating this request to the instructor for review."
    else:
        decision_line = "I’m unable to process this request at this time."

    email_args = {
        "student_id": email.get("student_id", ""),
        "subject": subject,
        "decision": decision_line,
        "rationale": rationale,
        "next_steps": next_steps
    }

    p2 = f"""{SYSTEM_INSTRUCTION}

TURN 2/3 - DRAFT EMAIL
Policy: {policy}
{memory_block}
{email_block}

You MUST end with exactly one tool call:
Action: draft_email[{json.dumps(email_args)}]
"""
    out2 = call_llm(p2).strip()
    trace.append(out2)
    if verbose:
        print("\n--- Step 3 Draft Email ---\n", out2[:300])

    tool2, arg2 = _last_action(out2)
    email_text_final = ""
    if tool2 == "draft_email":
        email_text_final = TOOLS["draft_email"](arg2)

    # Fallback: if LLM produced invalid JSON, draft deterministically
    if (not email_text_final) or ("error:" in email_text_final.lower()):
        email_text_final = TOOLS["draft_email"](json.dumps(email_args))

    # ---------------------------
    # Assemble result
    # ---------------------------
    result = {
        "action": action,
        "rationale": rationale,
        "next_steps": next_steps,
        "email": email_text_final,
        # Optional: include audit flag for debugging
        "audit": {
            "has_memory": has_memory,
            "prior_extension_confirmed": bool(prior_ext),
            "policy_key_used": arg1
        }
    }
    return result, trace


## STATELESS

In [34]:
# -------------------- THE EXPERIMENT --------------------
print("\n" + "="*70)
print("RUN 1) ReAct WITHOUT Memory (STATELESS)")
print("="*70)
print("\n✓ Agent sees: email + policies")
print("✗ Agent doesn't see: student history\n")

res_no_mem, trace_no_mem = react_agent(email_followup, memory_context=None, verbose=True)

print("\n STATELESS RESULT:")
print(json.dumps(res_no_mem, indent=2))



RUN 1) ReAct WITHOUT Memory (STATELESS)

✓ Agent sees: email + policies
✗ Agent doesn't see: student history


--- Step 1 Policy LookUp ---
 Thought: The current email indicates the student is requesting an extension for assignment A2 due to needing more time.

Action: kb_lookup[extension_policy]

--- Step 2 Decision ---
 {
  "action": "ASK_CLARIFICATION",
  "rationale": "The email suggests a repeat request, but the memory indicates no prior history. The student's previous requests were denied.",
  "next_steps": "Reach out to the instructor for further clarification on why the extension was not granted previously."
}

--- Step 3 Draft Email ---
 Thought: Based on the current email and policy, it appears that this is a repeat request due to the language used (again) and lack of verified history suggesting previous denial.

Action: draft_email[{"student_id": "22k-1234", "subject": "A2 \u2014 Extension Request Update", "decision": "I need one 

 STATELESS RESULT:
{
  "action": "ASK_CLARI

## Stateful agent

In [35]:
print("\n" + "="*70)
print("RUN 2) ReAct WITH Memory (STATEFUL)")
print("="*70)

history_json = json.dumps(mem.history("22k-1234"), indent=2)
print("\n✓ Agent sees: email + policies + HISTORY")
print("✓ Agent knows: prior events")
print("\nHistory:")
print(history_json)

print()

res_mem, trace_mem = react_agent(email_followup, memory_context=history_json, verbose=True)

print("\n STATEFUL RESULT:")
print(json.dumps(res_mem, indent=2))




RUN 2) ReAct WITH Memory (STATEFUL)

✓ Agent sees: email + policies + HISTORY
✓ Agent knows: prior events

History:
[
  {
    "intent": "extension",
    "assignment": "A2",
    "message_id": "m1",
    "action": "GRANT_EXTENSION"
  }
]


--- Step 1 Policy LookUp ---
 Thought: The current situation is a repeat request for an extension due to late submission.
Action: ASK_CLARIFICATION

--- Step 2 Decision ---
 {
  "action": "ESCALATE",
  "rationale": "The student has previously requested an extension (m1), indicating a repeat request. Given the repeated nature of this request and the urgency stated in the email, escalation is necessary.",
  "next_steps": "Inform the instructor that there's a repeat reques

--- Step 3 Draft Email ---
 Action: draft_email[{"student_id": "22k-1234", "subject": "A2 \u2014 Extension Request Update", "decision": "I\u2019m escalating this request to the instructor for review.", "rationale": "The student has previously requested an extension (m1), indicating a r

# -------------------- SIDE-BY-SIDE COMPARISON --------------------

In [36]:
print("\n" + "="*70)
print("SIDE-BY-SIDE COMPARISON")
print("="*70)

print("\nACTION:")
print(f"  Stateless: {res_no_mem.get('action')}")
print(f"  Stateful:  {res_mem.get('action')}")
print(f"  → Same? {res_no_mem.get('action') == res_mem.get('action')}")

print("\nRATIONALE (first 100 chars):")
print(f"  Stateless: {str(res_no_mem.get('rationale',''))[:100]}...")
print(f"  Stateful:  {str(res_mem.get('rationale',''))[:100]}...")
print(f"  → Same? {res_no_mem.get('rationale') == res_mem.get('rationale')}")

print("\n" + "="*70)
print("KEY INSIGHT")
print("="*70)

if res_no_mem.get('action') != res_mem.get('action'):
    print("✓ MEMORY CHANGED THE DECISION!")
    print(f"  Without memory: {res_no_mem.get('action')}")
    print(f"  With memory:    {res_mem.get('action')}")
else:
    print("✓ Same action, but REASONING is different")
    if res_no_mem.get('rationale') != res_mem.get('rationale'):
        print("  - Stateless inferred from policy")
        print("  - Stateful grounded in actual history")

print("\n→ Conclusion: Context matters! Memory enables policy enforcement based on history.")


SIDE-BY-SIDE COMPARISON

ACTION:
  Stateless: ASK_CLARIFICATION
  Stateful:  ESCALATE
  → Same? False

RATIONALE (first 100 chars):
  Stateless: I do not have access to prior history in this run. The email suggests a repeat request, but the memo...
  Stateful:  The student has previously requested an extension (m1), indicating a repeat request. Given the repea...
  → Same? False

KEY INSIGHT
✓ MEMORY CHANGED THE DECISION!
  Without memory: ASK_CLARIFICATION
  With memory:    ESCALATE

→ Conclusion: Context matters! Memory enables policy enforcement based on history.
